# 02 从全连接到卷积

## 目标
对照 MLP 与 CNN 的参数量与特征形状；能默写本课 CNN 的尺寸账本。

## 会错的直觉
「卷积会先翻转核再滑窗。」——PyTorch 的 conv2d 是互相关，不翻转。

本课 CNN：`Conv 1→16 k=5 p=2` → ReLU → Pool2 → `Conv 16→32 k=5 p=2` → ReLU → Pool2 → `Linear(32*7*7, 10)`。

尺寸：`[B,1,28,28] → [B,16,14,14] → [B,32,7,7] → [B,10]`。

In [ ]:
from mnist_lab.models import build_model
from mnist_lab.shapes import trace_shapes, format_trace, cnn_expected_shapes
cnn = build_model('cnn')
rows = trace_shapes(cnn, batch=2)
print(format_trace(rows))
expect = dict(cnn_expected_shapes(2))
got = dict(rows)
for name, shape in expect.items():
    assert got[name] == shape, (name, got.get(name), shape)
print('尺寸账本核对通过')

In [ ]:
mlp = build_model('mlp')
def nparams(m):
    return sum(p.numel() for p in m.parameters())
print('mlp', nparams(mlp), 'cnn', nparams(cnn))
assert nparams(cnn) < nparams(mlp)

互相关对照：NumPy 与 `F.conv2d` 应对齐。

In [ ]:
import numpy as np, torch, torch.nn.functional as F
from mnist_lab.numpy_conv import conv2d_numpy
img = np.random.randn(1, 1, 8, 8).astype(np.float32)
ker = np.random.randn(1, 1, 3, 3).astype(np.float32)
d = np.max(np.abs(conv2d_numpy(img, ker, padding=1) - F.conv2d(torch.tensor(img), torch.tensor(ker), padding=1).numpy()))
print('diff', d)
assert d < 1e-4

Dropout CNN 前向形状仍是 `[B,10]`。

In [ ]:
y = build_model('cnn_dropout')(torch.zeros(3, 1, 28, 28))
print(tuple(y.shape))
assert y.shape == (3, 10)

## 思考题
1. 为什么 CNN 参数更少，却通常更适合图像？
2. padding=2、kernel=5 时，池化前空间尺寸为什么仍是 28？
3. 若去掉两层 MaxPool，全连接输入还是 `32*7*7` 吗？